# Gold Layer: Data Modeling and Dimensional Tables

This notebook demonstrates the creation of the gold layer in a Lakehouse architecture using Databricks, PySpark, and Delta Lake. The gold layer consists of curated fact and dimension tables, optimized for analytics and business intelligence workloads.


---

## 1. Schema Creation <a name="schema-creation"></a>

Create the `gold` schema if it does not already exist.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

---

## 2. Fact and Dimension Table Creation <a name="fact-dim-creation"></a>

### Trips Fact Table <a name="trips-fact"></a>

Transform the curated trips data from the silver layer into a fact table. Calculate trip duration and select relevant columns for analytics.

In [0]:
from pyspark.sql.functions import col

gold_trips = (

    spark.table("silver.trips")
    .select(
        col("ride_id"),
        col("rideable_type"),
        col("started_at"),
        col("ended_at"),
        col("start_station_id"),
        col("end_station_id"),
        col("rider_id"),
        (col("ended_at").cast("long") - col("started_at").cast("long")).alias("duration"
    )
    )
)
gold_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.ft_trips")


---

### Rider Dimension Table <a name="rider-dim"></a>

Create a dimension table for riders, including personal and account information.

In [0]:
gold_dim_rider = (
    spark.table("silver.riders")
    .select(
        col("rider_id"),
        col("first_name"),
        col("last_name"),
        col("address"),
        col("birthday"),
        col("account_start_date"),
        col("account_end_date"),
        col("is_member")
    )
)

gold_dim_rider.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_rider")

---

### Station Dimension Table <a name="station-dim"></a>

Create a dimension table for stations, including location details.

In [0]:
gold_dim_station = (
    spark.table("silver.stations")
    .select(
        col("station_id"),
        col("station_name"),
        col("latitude"),
        col("longitude")
    )
)

gold_dim_station.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_station")

---

### Payments Fact Table <a name="payments-fact"></a>

Transform the curated payments data from the silver layer into a fact table, including payment details.

In [0]:
gold_fact_payments = (
    spark.table("silver.payments")
    .select(
        col("payment_id"),
        col("payment_date"),
        col("amount"),
        col("account_number")
    )
)

gold_fact_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.ft_payments")

---

### Date Dimension Table <a name="date-dim"></a>

Build a date dimension table by extracting unique dates from trips and payments, and enriching with calendar attributes.

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, dayofweek

dates_df = (
    spark.table("silver.trips")
        .select(col("started_at").cast("date").alias("date"))
    .union(
        spark.table("silver.trips")
            .select(col("ended_at").cast("date").alias("date"))
    )
    .union(
        spark.table("silver.payments")
            .select(col("payment_date").alias("date"))
    )
    .distinct()
)

gold_dim_date = (
    dates_df
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_of_week", dayofweek("date"))
)

gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_date")